# SWIFT vs BRICS Currency Share Analysis
**Source:** SWIFT RMB Tracker PDFs

This notebook extracts the **Global payments by currency (%)** table from each PDF, builds a month-level dataset, aggregates to quarters and answers the 4 required questions.

> Note: SWIFT tracker tables typically include CNY (RMB) but not a combined BRICS R5 series. Therefore we use **R5 proxy = CNY share** and mention this assumption in the report.


In [6]:
import glob

PDF_GLOB = "/Users/santoshd/Desktop/UE/DSB/Data-Science-Final-Project/data/RAW/Swift/rmb-tracker_*.pdf"

pdfs = glob.glob(PDF_GLOB)
print("PDF files found:", len(pdfs))
print(pdfs[:3])


PDF files found: 14
['/Users/santoshd/Desktop/UE/DSB/Data-Science-Final-Project/data/RAW/Swift/rmb-tracker_september-2025.pdf', '/Users/santoshd/Desktop/UE/DSB/Data-Science-Final-Project/data/RAW/Swift/rmb-tracker_march-2025.pdf', '/Users/santoshd/Desktop/UE/DSB/Data-Science-Final-Project/data/RAW/Swift/rmb-tracker_august-2025.pdf']


In [8]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
SWIFT vs BRICS Currency Share Analysis (from SWIFT RMB Tracker PDFs)

This script:
1) Extracts the "Global payments by currency (%)" table from SWIFT RMB Tracker PDFs
2) Builds a month-level currency share dataset
3) Aggregates to quarters
4) Answers 4 questions:
   Q1: USD vs R5 (proxy=CNY) growth in share
   Q2: Plot USD/EUR/R5 trends across quarters
   Q3: Compute Herfindahl-Hirschman Index (HHI) each quarter
   Q4: Find which quarter had the largest shift in USD share

Your project folder structure:
- PDFs are in: data/RAW/Swift/
- Outputs will be saved to: data/outputs/Swift/
"""

import os, re, glob, math
from pathlib import Path
import pandas as pd

try:
    import pdfplumber
except ImportError as e:
    raise SystemExit("Missing dependency: pdfplumber. Install with: pip install pdfplumber") from e

import matplotlib.pyplot as plt


# -----------------------------
# CONFIG (ROOT + PATHS)
# -----------------------------
# Detect project root reliably: must contain BOTH data/ and src/
REPO_ROOT = Path.cwd().resolve()
while not ((REPO_ROOT / "data").exists() and (REPO_ROOT / "src").exists()):
    if REPO_ROOT == REPO_ROOT.parent:
        raise SystemExit(
            "Could not detect project root.\n"
            "Please set REPO_ROOT manually like:\n"
            "REPO_ROOT = Path('/Users/santoshd/Desktop/UE/DSB/Data-Science-Final-Project')"
        )
    REPO_ROOT = REPO_ROOT.parent

print("✅ Repo root detected:", REPO_ROOT)

# PDFs folder
PDF_GLOB = str(REPO_ROOT / "data" / "RAW" / "Swift" / "rmb-tracker_*.pdf")

# Sanity check: confirm PDFs exist
_found_pdfs = glob.glob(PDF_GLOB)
print("✅ PDF count found:", len(_found_pdfs))
if len(_found_pdfs) == 0:
    raise SystemExit(
        f"No PDFs found.\nChecked glob: {PDF_GLOB}\n"
        f"Expected folder: {REPO_ROOT / 'data' / 'RAW' / 'Swift'}"
    )

# ✅ Outputs folder requested by you
OUT_DIR = REPO_ROOT / "data" / "outputs" / "Swift"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Output files
OUT_MONTHLY_CSV = OUT_DIR / "swift_currency_share_monthly_extracted.csv"
OUT_QUARTERLY_CSV = OUT_DIR / "swift_currency_share_quarterly.csv"


MONTH_MAP = {
    "january": 1, "february": 2, "march": 3, "april": 4,
    "may": 5, "june": 6, "july": 7, "august": 8,
    "september": 9, "october": 10, "november": 11, "december": 12
}

CURRENCY_CODES_KEEP = {
    "USD","EUR","GBP","JPY","CNY","AUD","CAD","CHF","HKD","SGD","SEK","NOK","DKK",
    "INR","RUB","BRL","ZAR","NZD","MXN","TRY","PLN","THB"
}


# -----------------------------
# HELPERS
# -----------------------------
def filename_to_date(pdf_name: str):
    """Convert rmb-tracker_month-year.pdf -> first day of that month."""
    m = re.search(r"rmb-tracker_([a-z]+)-(\d{4})", pdf_name.lower())
    if not m:
        return None
    mon = m.group(1)
    year = int(m.group(2))
    if mon not in MONTH_MAP:
        return None
    return pd.Timestamp(year, MONTH_MAP[mon], 1)


def extract_currency_shares(pdf_path: Path):
    """
    Extract currency share pairs like 'USD 46.71' from the SWIFT tracker page text.
    The table is usually on page 3 (index 2).
    """
    with pdfplumber.open(str(pdf_path)) as pdf:
        pages_to_try = [2, 1, 3, 0]  # try likely pages first
        text = ""
        for idx in pages_to_try:
            if idx < len(pdf.pages):
                t = pdf.pages[idx].extract_text() or ""
                if "USD" in t and "EUR" in t and "%" in t:
                    text = t
                    break

        # fallback: read all pages
        if not text:
            text = "\n".join([(p.extract_text() or "") for p in pdf.pages])

    pairs = re.findall(r"\b([A-Z]{3})\s+(\d{1,2}\.\d{2})\b", text)
    data = {}
    for cur, val in pairs:
        if cur in CURRENCY_CODES_KEEP and cur not in data:
            data[cur] = float(val)

    return data


def compute_hhi(row, share_cols):
    """HHI on [0..1] scale using shares in percentage."""
    vals = [row[c] for c in share_cols if pd.notna(row.get(c))]
    return sum((v/100.0)**2 for v in vals)


# -----------------------------
# MAIN
# -----------------------------
def main():
    pdf_files = [Path(p) for p in glob.glob(PDF_GLOB)]
    pdf_files = sorted(pdf_files, key=lambda p: p.name)

    if not pdf_files:
        raise SystemExit(f"No PDFs found with glob: {PDF_GLOB}")

    rows = []
    for p in pdf_files:
        dt = filename_to_date(p.name)
        if dt is None:
            continue
        shares = extract_currency_shares(p)
        shares["date"] = dt
        shares["pdf"] = p.name
        rows.append(shares)

    df = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)

    # Currency share columns (3-letter codes)
    share_cols = [c for c in df.columns if re.fullmatch(r"[A-Z]{3}", str(c))]

    # BRICS R5 proxy:
    # In SWIFT tracker tables, the only BRICS currency consistently present is CNY (RMB).
    # So we use R5_proxy = CNY (and you can note this assumption in the report).
    df["R5_proxy"] = df["CNY"]

    # Compute HHI monthly and quarterly
    df["quarter"] = df["date"].dt.to_period("Q").astype(str)
    df["HHI_month"] = df.apply(lambda r: compute_hhi(r, share_cols), axis=1)

    q = df.groupby("quarter").agg(
        USD_mean=("USD", "mean"),
        EUR_mean=("EUR", "mean"),
        R5_mean=("R5_proxy", "mean"),
        HHI=("HHI_month", "mean"),
        USD_start=("USD", "first"),
        USD_end=("USD", "last"),
    ).reset_index()
    q["USD_shift"] = q["USD_end"] - q["USD_start"]

    # -----------------------------
    # Q1: USD vs R5 growth
    # -----------------------------
    usd_growth = df["USD"].iloc[-1] - df["USD"].iloc[0]
    r5_growth = df["R5_proxy"].iloc[-1] - df["R5_proxy"].iloc[0]

    # -----------------------------
    # Q4: quarter with max USD shift
    # -----------------------------
    max_shift_row = q.loc[q["USD_shift"].abs().idxmax()]

    # Save tables
    df.to_csv(OUT_MONTHLY_CSV, index=False)
    q.to_csv(OUT_QUARTERLY_CSV, index=False)

    # -----------------------------
    # Q2: Plots
    # -----------------------------
    plt.figure()
    plt.plot(q["quarter"], q["USD_mean"], marker="o")
    plt.xticks(rotation=45)
    plt.title("USD share trend (quarterly mean)")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "usd_share_quarterly.png", dpi=200)
    plt.close()

    plt.figure()
    plt.plot(q["quarter"], q["EUR_mean"], marker="o")
    plt.xticks(rotation=45)
    plt.title("EUR share trend (quarterly mean)")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "eur_share_quarterly.png", dpi=200)
    plt.close()

    plt.figure()
    plt.plot(q["quarter"], q["R5_mean"], marker="o")
    plt.xticks(rotation=45)
    plt.title("R5 proxy (CNY/RMB) share trend (quarterly mean)")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "r5_proxy_share_quarterly.png", dpi=200)
    plt.close()

    # -----------------------------
    # Print answers
    # -----------------------------
    print("\n==============================")
    print("SWIFT vs BRICS Currency Share")
    print("==============================\n")

    print("Q1) Which currency (USD vs R5) shows highest growth in share?")
    print(f"   USD growth (last-first): {usd_growth:.2f} percentage points")
    print(f"   R5 proxy growth (CNY last-first): {r5_growth:.2f} percentage points")
    if abs(usd_growth) > abs(r5_growth):
        print("   -> USD shows higher absolute growth/change (and remains dominant).\n")
    else:
        print("   -> R5 proxy (CNY) shows higher absolute growth/change.\n")

    print("Q2) Trend plots saved to:")
    print(f"   - {OUT_DIR / 'usd_share_quarterly.png'}")
    print(f"   - {OUT_DIR / 'eur_share_quarterly.png'}")
    print(f"   - {OUT_DIR / 'r5_proxy_share_quarterly.png'}\n")

    print("Q3) Herfindahl Index (HHI) each quarter saved in:")
    print(f"   - {OUT_QUARTERLY_CSV}\n")

    print("Q4) Which quarter had the largest shift in USD share?")
    print(f"   Quarter: {max_shift_row['quarter']}")
    print(f"   USD start: {max_shift_row['USD_start']:.2f}%, USD end: {max_shift_row['USD_end']:.2f}%")
    print(f"   USD shift: {max_shift_row['USD_shift']:.2f} percentage points\n")

    print("✅ Outputs saved to:")
    print(f"   Folder: {OUT_DIR}")
    print(f"   Monthly table:  {OUT_MONTHLY_CSV}")
    print(f"   Quarterly table:{OUT_QUARTERLY_CSV}")
    print("")


if __name__ == "__main__":
    main()


✅ Repo root detected: /Users/santoshd/Desktop/UE/DSB/Data-Science-Final-Project
✅ PDF count found: 14

SWIFT vs BRICS Currency Share

Q1) Which currency (USD vs R5) shows highest growth in share?
   USD growth (last-first): -0.30 percentage points
   R5 proxy growth (CNY last-first): -1.14 percentage points
   -> R5 proxy (CNY) shows higher absolute growth/change.

Q2) Trend plots saved to:
   - /Users/santoshd/Desktop/UE/DSB/Data-Science-Final-Project/data/outputs/Swift/usd_share_quarterly.png
   - /Users/santoshd/Desktop/UE/DSB/Data-Science-Final-Project/data/outputs/Swift/eur_share_quarterly.png
   - /Users/santoshd/Desktop/UE/DSB/Data-Science-Final-Project/data/outputs/Swift/r5_proxy_share_quarterly.png

Q3) Herfindahl Index (HHI) each quarter saved in:
   - /Users/santoshd/Desktop/UE/DSB/Data-Science-Final-Project/data/outputs/Swift/swift_currency_share_quarterly.csv

Q4) Which quarter had the largest shift in USD share?
   Quarter: 2025Q4
   USD start: 47.79%, USD end: 46.71%
   

In [4]:
%pip install pdfplumber pandas matplotlib

  Using cached cryptography-46.0.3-cp311-abi3-macosx_10_9_universal2.whl.metadata (5.7 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pycparser-2.23-py3-none-any.whl.metadata (993 bytes)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 10.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 10.9 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 11.0 MB/s eta 0:00:00a 0:00:01
Using cached cryptography-46.0.3-cp311-abi3-macosx_10_9_universal2.whl (7.2 MB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 10.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 10.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 10.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 11.1 MB/s eta 0:00:00a 0:00:01
Using cached pycparser-2.23-py3-none-any.

In [2]:
import glob
glob.glob("data/raw/swift/rmb-tracker_*.pdf")[:5]


[]